In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

In [ ]:
#read in the data - saved as a csv file at a checkpoint
rgdp_county_allindustry = pd.read_csv('data/rgdp_county_allindustry.csv')
rgdp_county_allindustry.tail()

In [ ]:
# Select year = 2021
rgdp_county_allindustry_2021 = rgdp_county_allindustry[rgdp_county_allindustry['year'] == 2021].copy()




In [ ]:
#plot a histogram of the data with kde overlay
sns.histplot(rgdp_county_allindustry_2021['value'], kde=True)

In [ ]:
rgdp_county_allindustry_2021['value'].value_counts(bins=20)



In [ ]:
#show the 10 counties with hightest RGDP
rgdp_county_allindustry_2021.nlargest(10, 'value')

In [ ]:
# make a list with the 9 bay area counties, with ",CA" appended to each county name
bay_area_counties = ['Alameda, CA', 'Contra Costa, CA', 'Marin, CA', 'Napa, CA', 'San Francisco, CA', 'San Mateo, CA', 'Santa Clara, CA', 'Solano, CA', 'Sonoma, CA']



In [ ]:
#Show the RGDP for the 9 bay area counties
rgdp_county_allindustry_2021[rgdp_county_allindustry_2021['GeoName'].isin(bay_area_counties)]


In [ ]:
# Lets rank the counties by RGDP and make a ranking column
rgdp_county_allindustry_2021['rank'] = rgdp_county_allindustry_2021['value'].rank(ascending=False)

In [ ]:
# Show bay area counties with their ranks
rgdp_county_allindustry_2021[rgdp_county_allindustry_2021['GeoName'].isin(bay_area_counties)][['GeoName', 'value', 'rank']].sort_values('rank')

In [ ]:
# Calculate the combined GDP of the Bay Area
bay_area_gdp = rgdp_county_allindustry_2021[rgdp_county_allindustry_2021['GeoName'].isin(bay_area_counties)]['value'].sum()
print(f"Combined Bay Area GDP: ${bay_area_gdp:,.2f} ")


In [ ]:
#can you make a table with bins of a histogram for RGDP
#make a histogram with 10 bins
rgdp_county_allindustry_2021['value'].hist(bins=10)
# can you report number of counties in each bin
rgdp_county_allindustry_2021['value'].value_counts(bins=10)


In [ ]:
#calculate the mean RGDP across all counties
rgdp_county_allindustry_2021['value'].mean()


In [ ]:
#format with commas and 2 decimal places
'{:,.2f}'.format(rgdp_county_allindustry_2021['value'].mean())

In [ ]:
#calculate the median RGDP across all counties
rgdp_county_allindustry_2021['value'].median()


In [ ]:
'{:,.2f}'.format(rgdp_county_allindustry_2021['value'].median())

In [ ]:
diff = rgdp_county_allindustry_2021['value'].mean() - rgdp_county_allindustry_2021['value'].median()
'{:,.2f}'.format(diff)

## Alternative: Interactive Map with Plotly

For a more detailed choropleth map with actual county boundaries, you can use Plotly Express with GeoJSON data. This requires loading county boundary data from a shapefile or GeoJSON source.

In [ ]:
# Alternative visualization: Bar chart of Bay Area counties by RGDP
bay_area_data = rgdp_county_allindustry_2021[rgdp_county_allindustry_2021['GeoName'].isin(bay_area_counties)].copy()
bay_area_data = bay_area_data.sort_values('value', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(bay_area_data['GeoName'].str.replace(', CA', ''), bay_area_data['value'], 
               color=plt.cm.YlOrRd(bay_area_data['value']/bay_area_data['value'].max()))

ax.set_xlabel('RGDP (millions $)', fontsize=12)
ax.set_title('Bay Area Counties - Real GDP (2021)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3, linestyle='--')

# Add value labels on bars
for i, (idx, row) in enumerate(bay_area_data.iterrows()):
    ax.text(row['value'], i, f" ${row['value']:,.0f}M", 
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Add FIPS codes manually for Bay Area counties
bay_area_fips = {
    'Alameda, CA': '06001',
    'Contra Costa, CA': '06013',
    'Marin, CA': '06041',
    'Napa, CA': '06055',
    'San Francisco, CA': '06075',
    'San Mateo, CA': '06081',
    'Santa Clara, CA': '06085',
    'Solano, CA': '06095',
    'Sonoma, CA': '06097'
}

# Create a copy of Bay Area data and add FIPS codes
bay_area_with_fips = rgdp_county_allindustry_2021[rgdp_county_allindustry_2021['GeoName'].isin(bay_area_counties)].copy()
bay_area_with_fips['fips'] = bay_area_with_fips['GeoName'].map(bay_area_fips)

bay_area_with_fips[['GeoName', 'value', 'rank', 'fips']]

In [ ]:
# Create interactive Plotly choropleth map with FIPS codes
import plotly.express as px

fig = px.choropleth(
    bay_area_with_fips,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
    locations='fips',
    color='value',
    color_continuous_scale="YlOrRd",
    hover_name='GeoName',
    hover_data={'value': ':,.0f', 'rank': ':.0f', 'fips': False},
    labels={'value': 'RGDP (millions $)'},
    title='Bay Area Counties - Real GDP (2021)',
    scope="usa"
)

# Zoom to Bay Area
fig.update_geos(
    center=dict(lon=-122.2, lat=37.8),
    projection_scale=20
)

fig.update_layout(height=600, margin={"r":0,"t":50,"l":0,"b":0})
fig.show()

### About Plotly Maps

Yes! Plotly has excellent built-in support for US geographic data:

- **`plotly.express.choropleth`** can display US states and counties using FIPS codes
- Plotly includes a built-in GeoJSON file with all US county boundaries
- You need county FIPS codes (5-digit identifiers) to map the data
- The map is interactive - you can zoom, pan, and hover for details

California county FIPS codes start with "06" (California's state code) followed by 3 digits for each county.